# Nuclei Segmentation with U-Net - Demo Notebook

This notebook demonstrates the trained U-Net model for nuclei segmentation.

**Contents:**
1. Load trained model
2. Run inference on sample images
3. Visualize predictions vs ground truth
4. Display evaluation metrics

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import torch
from PIL import Image

from model import UNet
from dataset import get_dataloaders, get_transforms
from evaluate import dice_coefficient, iou_score

# Set matplotlib style
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Load Trained Model

In [ ]:
# Setup device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model
model_path = '../outputs/best_model.pth'
model = UNet(in_channels=3, out_channels=1).to(device)

checkpoint = torch.load(model_path, map_location=device)
if 'model_state_dict' in checkpoint:
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nModel checkpoint info:")
    print(f"  Epoch: {checkpoint.get('epoch', 'unknown')}")
    print(f"  Validation Dice: {checkpoint.get('val_dice', 'unknown')}")
else:
    model.load_state_dict(checkpoint)

model.eval()
print("\n✓ Model loaded successfully!")

## 2. Load Validation Data

In [ ]:
# Load validation set
data_dir = '../data/stage1_train'
_, val_loader = get_dataloaders(
    data_dir=data_dir,
    batch_size=8,
    img_size=256,
    val_split=0.2,
    num_workers=0
)

print(f"Validation set size: {len(val_loader.dataset)} images")

## 3. Visualize Sample Predictions

In [ ]:
def denormalize(image):
    """Denormalize image from model input format to display format."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    image = image * std + mean
    image = torch.clamp(image, 0, 1)
    return image

def show_predictions(model, dataloader, device, num_samples=5):
    """Display predictions for a few samples."""
    model.eval()
    
    # Get one batch
    images, masks = next(iter(dataloader))
    images = images.to(device)
    masks = masks.to(device)
    
    # Run inference
    with torch.no_grad():
        logits = model(images)
        predictions = torch.sigmoid(logits)
    
    # Convert to numpy and show
    num_samples = min(num_samples, images.size(0))
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(num_samples):
        # Denormalize image for display
        img = denormalize(images[idx].cpu()).permute(1, 2, 0).numpy()
        gt_mask = masks[idx, 0].cpu().numpy()
        pred_mask = predictions[idx, 0].cpu().numpy()
        pred_binary = (pred_mask > 0.5).astype(float)
        
        # Calculate metrics
        dice = dice_coefficient(
            predictions[idx:idx+1],
            masks[idx:idx+1],
            threshold=0.5
        )
        iou = iou_score(
            predictions[idx:idx+1],
            masks[idx:idx+1],
            threshold=0.5
        )
        
        # Plot
        axes[idx, 0].imshow(img)
        axes[idx, 0].set_title('Input Image', fontweight='bold')
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(gt_mask, cmap='gray')
        axes[idx, 1].set_title('Ground Truth', fontweight='bold')
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(pred_mask, cmap='hot', vmin=0, vmax=1)
        axes[idx, 2].set_title('Prediction (Probability)', fontweight='bold')
        axes[idx, 2].axis('off')
        
        axes[idx, 3].imshow(pred_binary, cmap='gray')
        axes[idx, 3].set_title(f'Prediction (Binary)\nDice: {dice:.3f} | IoU: {iou:.3f}', 
                              fontweight='bold')
        axes[idx, 3].axis('off')
    
    plt.tight_layout()
    plt.savefig('../outputs/demo_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

# Show predictions
show_predictions(model, val_loader, device, num_samples=5)

## 4. Quantitative Evaluation

In [ ]:
def evaluate_full_dataset(model, dataloader, device):
    """Evaluate model on entire dataset."""
    model.eval()
    
    dice_scores = []
    iou_scores = []
    
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            
            logits = model(images)
            predictions = torch.sigmoid(logits)
            
            # Calculate metrics for each image
            for i in range(images.size(0)):
                dice = dice_coefficient(predictions[i:i+1], masks[i:i+1])
                iou = iou_score(predictions[i:i+1], masks[i:i+1])
                dice_scores.append(dice)
                iou_scores.append(iou)
    
    return dice_scores, iou_scores

# Evaluate
print("Evaluating on validation set...")
dice_scores, iou_scores = evaluate_full_dataset(model, val_loader, device)

# Print summary statistics
print("\n" + "="*60)
print("EVALUATION METRICS")
print("="*60)
print(f"Number of validation samples: {len(dice_scores)}")
print(f"\nDice Coefficient:")
print(f"  Mean:   {np.mean(dice_scores):.4f}")
print(f"  Std:    {np.std(dice_scores):.4f}")
print(f"  Min:    {np.min(dice_scores):.4f}")
print(f"  Max:    {np.max(dice_scores):.4f}")
print(f"  Median: {np.median(dice_scores):.4f}")
print(f"\nIoU Score:")
print(f"  Mean:   {np.mean(iou_scores):.4f}")
print(f"  Std:    {np.std(iou_scores):.4f}")
print(f"  Min:    {np.min(iou_scores):.4f}")
print(f"  Max:    {np.max(iou_scores):.4f}")
print(f"  Median: {np.median(iou_scores):.4f}")
print("="*60)

## 5. Visualize Metric Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dice distribution
axes[0].hist(dice_scores, bins=30, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(dice_scores), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(dice_scores):.3f}')
axes[0].set_xlabel('Dice Coefficient', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Dice Coefficients', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# IoU distribution
axes[1].hist(iou_scores, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].axvline(np.mean(iou_scores), color='red', linestyle='--', 
                linewidth=2, label=f'Mean: {np.mean(iou_scores):.3f}')
axes[1].set_xlabel('IoU Score', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of IoU Scores', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Display Training Curves

In [ ]:
# Load and display training curves if they exist
training_curves_path = '../outputs/training_curves.png'
if Path(training_curves_path).exists():
    img = Image.open(training_curves_path)
    plt.figure(figsize=(12, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training History', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Training curves not found. Train the model first using train.py")

## Summary

This demo notebook shows:
- ✓ The trained U-Net model successfully segments nuclei in microscopy images
- ✓ Quantitative evaluation using Dice coefficient and IoU metrics
- ✓ Visual comparison of predictions vs ground truth
- ✓ Metric distributions showing model consistency

### Key Findings:
- The model performs well on most images with diverse imaging conditions
- Some challenging cases include overlapping nuclei and low-contrast images
- The combined BCE + Dice loss helps handle class imbalance effectively

### Next Steps:
- Try different thresholds for binary prediction
- Experiment with more augmentations for difficult cases
- Fine-tune on specific imaging modalities if needed